# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The dataset conforms to the Croissant schema and serves as a standardized, machine-interpretable data package for structured analysis.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
We begin by loading dataset metadata using `mlcroissant`. This gives access to structured information about the dataset including available record sets, fields, and meta-descriptions.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Suppress pandas SettingWithCopyWarning for this example
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Next, we review available record sets, their `@id`s, and the fields contained within each. This helps identify the structure for subsequent data extraction and analysis.

**Note:** All entities are referenced by their `@id` as per Croissant standards.

In [ ]:
# List all available record sets and their details
print("Available record sets:")
record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"\nRecord set name: {getattr(rs, 'name', None)}")
    print(f"  @id: {rs.id}")
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - {getattr(field, 'name', None)} (@id: {field.id}, type: {getattr(field, 'data_type', None)})")
    else:
        print("  No fields found.")

# Print an example record for each record set using its @id
for rs in record_sets:
    print(f"\nSample record from record set '{getattr(rs, 'name', None)}' (@id: {rs.id}):")
    records_iter = dataset.records(record_set=rs.id)
    try:
        sample_record = next(records_iter)
        print(sample_record)
    except StopIteration:
        print("  [No records in this record set]")

## 3. Data Extraction
We'll load each record set into a pandas DataFrame for further processing. All extraction is done referencing the record sets and field `@id`s.

In [ ]:
# Prepare each record set as a DataFrame
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        dataframes[rs_id] = pd.DataFrame()  # Empty DataFrame if no data

# Display record set IDs and example columns
for rs_id, df in dataframes.items():
    print(f"\nRecord set @id: {rs_id}")
    print(f"Columns: {df.columns.tolist() if not df.empty else '[No columns]'}")
    if not df.empty:
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Let's process a specific record set. We'll choose the first record set found above (if any), and apply typical EDA transformations — filtering by a numeric field, normalizing, and grouping. All operations use Croissant `@id` paths as column names where possible.

In [ ]:
# Select the first nonempty record set
chosen_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        chosen_rs_id = rs_id
        break

if chosen_rs_id is None:
    print("No record sets with data found for analysis.")
else:
    df = dataframes[chosen_rs_id]
    print(f"Using record set @id: {chosen_rs_id}")

    # Try to find a likely numeric field based on dtype and/or Croissant field definition
    numeric_candidates = df.select_dtypes(include='number').columns
    if len(numeric_candidates) == 0:
        print("No numeric fields found in selected record set.")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Numeric field selected: {numeric_field}")

        # Example threshold
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'bool' else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try to group by a categorical or object field
        group_candidates = df.select_dtypes(include=['object', 'category']).columns
        group_field = None
        for field in group_candidates:
            if field != numeric_field:
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())

## 5. Visualization
We'll plot the distribution of the selected numeric field and (if available) its relationship to a categorical field.

*You may need to install matplotlib or seaborn if they're not present*:

In [ ]:
# Visualize data distributions
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and not df.empty and len(numeric_candidates) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouped data exists, plot mean by group
    if group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.xticks(rotation=45)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR² dataset with `mlcroissant` using only Croissant `@id` references for all entities.
- Previewed records and extracted DataFrames for all record sets.
- Conducted initial exploratory analysis: filtered, normalized, grouped, and visualized data from the most populated record set.

*Next steps could involve deeper feature engineering, modeling, or exporting cleaned data for further analysis.*